In [2]:
# Read names of the mocules
names_dict = {}
with open("data/names_nocids.txt", "r") as file:
    for line in file:
        elements = line.split(" ")
        id = elements[-1]
        name = "".join(elements[:-1])
        name = name.strip()
        id = id.strip()

        if name not in list(names_dict.values()):
            names_dict[id] = name    

In [3]:
import requests
import re
import urllib.parse

def chebi_to_smiles(name):
    """
    Busca o primeiro match no ChEBI para 'name' e tenta extrair o SMILES
    da página HTML de detalhes da molécula.

    Retorna:
        - string SMILES, se encontrado
        - None, se não encontrar ou em caso de erro
    """
    query = urllib.parse.quote(name)
    url = f"https://www.ebi.ac.uk/chebi/searchId.do?searchString={query}"

    try:
        r = requests.get(url, timeout=10)
    except requests.RequestException:
        return None

    if r.status_code != 200:
        return None

    html = r.text

    # 1) Tentativa principal: <dt>SMILES</dt><dd>...</dd>
    m = re.search(r'SMILES</dt>\s*<dd>\s*([^<]+?)\s*</dd>', html, re.IGNORECASE)
    if m:
        smiles = m.group(1).strip()
        if smiles:
            return smiles

    # 2) Fallback: data-smiles="..."
    m = re.search(r'data-smiles="([^"]+)"', html, re.IGNORECASE)
    if m:
        smiles = m.group(1).strip()
        if smiles:
            return smiles

    return None


def name_to_mol(name):
    """
    Tenta converter nome → SMILES usando OPSIN.
    """
    query = urllib.parse.quote(name)
    url = f"https://opsin.ch.cam.ac.uk/opsin/{query}.smi"
    try:
        response = requests.get(url, timeout=10)
    except requests.RequestException:
        return None

    if response.status_code == 200:
        smiles = response.text.strip()
        return smiles if smiles else None
    else:
        return None


def cactus_name_to_smiles(name):
    """
    Tenta converter nome → SMILES usando o servidor CACTUS (NIH).
    """
    query = urllib.parse.quote(name)
    url = f"https://cactus.nci.nih.gov/chemical/structure/{query}/smiles"
    try:
        r = requests.get(url, timeout=10)
    except requests.RequestException:
        return None

    if r.status_code == 200:
        smiles = r.text.strip()
        return smiles if smiles else None
    return None


def get_smiles(name):
    """
    Tenta, em ordem:
    1) OPSIN
    2) ChEBI
    3) CACTUS

    Retorna o primeiro SMILES válido encontrado ou None.
    """
    for func in (name_to_mol, chebi_to_smiles, cactus_name_to_smiles):
        smiles = func(name)
        if smiles is not None:
            return smiles
        #else:
            #print(name)
    return None


# names_dict deve ser um dict {id: nome_da_molécula}
smiles_dict = {}
for mol_id, name in names_dict.items():
    smiles = get_smiles(name)
    if smiles is not None:
        smiles_dict[mol_id] = smiles


/usr/lib/python3/dist-packages/requests/__init__.py:87: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (4.0.0) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [5]:
names_dict

{'nocid1': '(2R)-1-(3,4-dihydroxyphenyl)-3-(2,4,6-trihydroxyphenyl)-propan-2-ol',
 'nocid2': '(2S)-1-(3,4-dihydroxyphenyl)-3-(2,4,6-trihydroxyphenyl)propan-2-ol',
 'nocid3': '(2S)-1-(3-hydroxyphenyl)-3-(2,4,6-trihydroxyphenyl)propan-2-ol',
 'nocid4': 'procollagentype1n-terminalpropeptide',
 'nocid5': 'typeIcollagencross-linkedc-telopeptide',
 'nocid6': 'Dehydroxychlorogenicacid',
 'nocid7': 'Dihydrocaffeoyl-glycerol',
 'nocid8': 'Dihydrochlorogenicacid',
 'nocid9': "3'-desmethylarctigenin",
 'nocid10': '(2R)-1-(3-hydroxyphenyl)-3-(2,4,6-trihydroxyphenyl)propan-2-ol',
 'nocid11': 'Kaempferol-3-sorphoroside',
 'nocid12': '(4R)-4-hydroxy-5-(3,4-dihydroxyphenyl)valericacid',
 'nocid13': '(4R)-4-hydroxy-5-(3-hydroxyphenyl)valericacid',
 'nocid14': '(4R)-5-(3,4-dihydroxyphenyl)-��-valerolactone',
 'nocid15': '(4R)-5-(3-hydroxyphenyl)-��-valerolactone',
 'nocid16': '(4S)-4-hydroxy-5-(3,4-dihydroxyphenyl)valericacid',
 'nocid17': '(4S)-4-hydroxy-5-(3-hydroxyphenyl)valericacid',
 'nocid18': '(4

In [4]:
smiles_dict

{'nocid1': 'OC=1C=C(C=CC1O)C[C@H](CC1=C(C=C(C=C1O)O)O)O',
 'nocid2': 'OC=1C=C(C=CC1O)C[C@@H](CC1=C(C=C(C=C1O)O)O)O',
 'nocid3': 'OC=1C=C(C=CC1)C[C@@H](CC1=C(C=C(C=C1O)O)O)O',
 'nocid7': 'C(CCC1=CC(O)=C(O)C=C1)(=O)C(O)C(O)CO',
 'nocid10': 'OC=1C=C(C=CC1)C[C@H](CC1=C(C=C(C=C1O)O)O)O',
 'nocid12': 'O[C@H](CCC(=O)O)CC1=CC(=C(C=C1)O)O',
 'nocid13': 'O[C@H](CCC(=O)O)CC1=CC(=CC=C1)O',
 'nocid16': 'O[C@@H](CCC(=O)O)CC1=CC(=C(C=C1)O)O',
 'nocid17': 'O[C@@H](CCC(=O)O)CC1=CC(=CC=C1)O'}

In [4]:
len(smiles_dict.keys()) 

9

In [6]:
# Append SMILES and IDs to the file previous generated by the R code
with open("data/smiles_nocid.txt", "w") as file:
    for id, smiles in smiles_dict.items():
        file.write(f"{smiles} {id}\n")

Check if those have CID.
Then search for the misssing ones manually, concatenate all smiles and do target prediction.